# Example: Simulating an Integer Systolic Array
In this notebook we will:
- Obtain a behavioral netlist of an integer systolic array (based on https://github.com/abarajithan11/axis-systolic-array) using Yosys
- Simulate a tile computation using an Arbolta design wrapper
- Perform a small fault injection campaign

Here is a look at the systolic array's top module which uses an AXI4-Stream interface:
```systemverilog
module int_systolic_array #(
    parameter int unsigned Rows,
    parameter int unsigned Cols,
    parameter int unsigned WidthOp0,
    parameter int unsigned WidthOp1,
    parameter int unsigned WidthAcc,
    parameter int unsigned LatencyMul,
    parameter int unsigned LatencyAcc,
) (
    input  logic                          clk_i,
    input  logic                          rst_ni,
    input  logic                          s_valid_i,
    input  logic                          s_last_i,
    input  logic                          m_ready_i,
    output logic                          s_ready_o,
    output logic                          m_valid_o,
    output logic                          m_last_o,
    input  logic [Rows-1:0][WidthOp0-1:0] s_op0_data_i,
    input  logic [Cols-1:0][WidthOp1-1:0] s_op1_data_i,
    output logic [Rows-1:0][WidthAcc-1:0] m_data_o
);
```

In [ ]:
import tempfile
from pathlib import Path

from pyosys import libyosys as ys  # type: ignore


def run_yosys(
    top_module: str,
    rtl_path: Path,
    synth_parameters: dict[str, int] | None = None,
    keep_hierarchy: bool = True,
    ignore_timing: bool = True,
    extra_passes: list[str] | None = None,
) -> str:
    """
    Helper for getting netlist from SystemVerilog design
    Returns raw JSON netlist
    """
    design = ys.Design()

    # Build load pass for SystemVerilog design
    load_pass = [f"read_slang {rtl_path} --top {top_module}"]
    if synth_parameters:
        params = [f"-G{p}={v}" for p, v in synth_parameters.items()]
        load_pass.append(" ".join(params))

    if keep_hierarchy:
        load_pass.append("--keep-hierarchy")

    if ignore_timing:
        load_pass.append("--ignore-timing")

    ys.run_pass(" ".join(load_pass), design)

    # Run other passes
    if extra_passes:
        [ys.run_pass(p, design) for p in extra_passes]

    # Get netlist
    netlist: str
    with tempfile.NamedTemporaryFile() as tmp:
        ys.run_pass(f"write_json {tmp.name}", design)
        netlist = Path(tmp.name).read_text("utf-8")

    return netlist

In [ ]:
from typing import Callable

import numpy as np
import polars as pl
from arbolta import HardwareDesign, PortConfig

# Helper
type EvalCallback = Callable[[HardwareDesign], None]


class SystolicArray:
    def __init__(
        self,
        rows: int,
        cols: int,
        width_op0: int,
        width_op1: int,
        width_acc: int,
        latency_mul: int,
        latency_acc: int,
        top_module: str,
        rtl_path: Path,
    ):
        self.rows = rows
        self.cols = cols
        self.width_op0 = width_op0
        self.width_op1 = width_op1
        self.latency_mul = latency_mul
        self.latency_acc = latency_acc

        # Load paramterized design into Yosys and export netlist
        netlist = run_yosys(
            top_module=top_module,
            rtl_path=rtl_path,
            synth_parameters={
                "Rows": rows,
                "Cols": cols,
                "WidthOp0": width_op0,
                "WidthOp1": width_op1,
                "WidthAcc": width_acc,
                "LatencyMul": latency_mul,
                "LatencyAcc": latency_acc,
            },
            extra_passes=[
                "setundef -zero",
                "flatten -scopename",
                "memory_map",
            ],
        )

        # Load design netlist into Arbolta for simulation
        self.design: HardwareDesign = HardwareDesign(
            netlist=netlist.encode("utf-8"),
            top_module=top_module,
            config={
                "clk_i": PortConfig(clock=True, polarity=1),
                "rst_ni": PortConfig(reset=True, polarity=0),
                "s_op0_data_i": PortConfig((1, self.rows), np.int_),
                "s_op1_data_i": PortConfig((1, self.cols), np.int_),
                "m_data_o": PortConfig((1, self.rows), np.int_),
            },
        )

        # Create dataframe of design cells
        module_info = self.design.netlist()["modules"][top_module]
        self.cell_df = pl.DataFrame(
            (
                {
                    "cell_name": name,
                    "cell_type": info["type"],
                    "scopename": info["attributes"].get("scopename"),
                    "net": net,
                    "port_direction": info["port_directions"].get(port_name),
                    "port_name": port_name,
                    "cell_src": info["attributes"].get("src"),
                }
                for name, info in module_info["cells"].items()
                for port_name, port_nets in info["connections"].items()
                for net in (n for n in set(map(int, port_nets)) if n not in {0, 1})
            ),
            infer_schema_length=None,
        )

    def tile_latency(self, k: int) -> int:
        """
        Calculate number of cycles for tile with inner-dimension `k`
        """
        return k + self.latency_mul + self.latency_acc + self.rows + 2 * self.cols - 3

    def compute_tile(
        self,
        x: np.ndarray[tuple[int, int], np.dtype[np.number]],
        k: np.ndarray[tuple[int, int], np.dtype[np.number]],
        eval_callback: EvalCallback | None = None,
    ) -> np.ndarray:
        """
        Run inputs through systolic array and compute a tile.
        Expects x: (K,R), k: (K,C) -> y: (C,R)
        """
        assert x.ndim == 2 and k.ndim == 2 and x.shape[0] == k.shape[0]
        K, R, C = *x.shape, k.shape[1]
        assert R == self.rows and C == self.cols

        results = np.zeros((C, R), dtype=self.design.ports.m_data_o.dtype)

        # Read results as we clock design, keep track of number of results read
        result_count = 0

        def step():
            nonlocal result_count

            if eval_callback:
                eval_callback(self.design)
            else:
                self.design.eval_clocked()

            if self.design.ports.m_valid_o:
                results[result_count] = self.design.ports.m_data_o
                result_count += 1

        # Pass inputs
        for i in range(K):
            while not self.design.ports.s_ready_o:
                step()

            self.design.ports.s_last_i = int(i == K - 1)
            self.design.ports.s_valid_i = 1
            self.design.ports.s_op0_data_i = x[i]
            self.design.ports.s_op1_data_i = k[i]

            step()

        self.design.ports.m_ready_i = 1
        self.design.ports.s_valid_i = 0
        self.design.ports.s_last_i = 0
        self.design.eval()

        while not self.design.ports.m_last_o:
            step()

        return results

In [ ]:
sa_wrapper = SystolicArray(
    rows=2,
    cols=2,
    width_op0=8,
    width_op1=8,
    width_acc=32,
    latency_acc=1,
    latency_mul=1,
    top_module="int_systolic_array",
    rtl_path=Path("hardware", "int_systolic_array.sv"),
)

In [ ]:
K = 100
op0_range = -(2 ** (sa_wrapper.width_op0 - 1)), 2 ** (sa_wrapper.width_op0 - 1)
op1_range = -(2 ** (sa_wrapper.width_op1 - 1)), 2 ** (sa_wrapper.width_op1 - 1)

a = np.random.randint(*op0_range, (K, sa_wrapper.rows))
b = np.random.randint(*op0_range, (K, sa_wrapper.cols))
golden = b.T @ a

In [ ]:
sa_wrapper.design.reset()
sa_wrapper.design.eval_reset_clocked()
result = sa_wrapper.compute_tile(a, b)

assert np.allclose(result, golden)

In [ ]:
from dataclasses import dataclass, field


class HangError(Exception):
    def __init__(self, max_cycles: int):
        self.max_cycles = max_cycles
        super().__init__(f"Design exceeded max cycles `{max_cycles}`")


@dataclass(slots=True)
class FaultInjectionHandle:
    net: int
    target_cycles: int
    max_cycles: int
    elapsed_cycles: int = field(init=False, default=0)

    def create_callback(self) -> EvalCallback:
        def callback(design: HardwareDesign):
            if self.elapsed_cycles >= self.max_cycles:
                raise HangError(max_cycles)

            # Insert bit-flip (SEU / transient fault)
            if self.elapsed_cycles == self.target_cycles:
                # Get reference to signal
                signal = design.signals[self.net]
                # Toggle bit/value
                signal.value = not signal.value

            self.elapsed_cycles += 1
            design.eval_clocked()

        return callback

In [ ]:
ff_nets = sa_wrapper.cell_df.filter(
    pl.col("cell_type") == "$dff",
    pl.col("port_direction") == "output",
)
ff_nets.select("cell_name", "net")

In [ ]:
import random

from tqdm.auto import tqdm

results = []
max_cycles = sa_wrapper.tile_latency(K)
for cell_name, net in tqdm(
    ff_nets.select("cell_name", "net").iter_rows(), total=len(ff_nets)
):
    target_cycles = random.randint(0, max_cycles)
    handle = FaultInjectionHandle(
        net=net, target_cycles=target_cycles, max_cycles=max_cycles
    )

    status: str
    try:
        sa_wrapper.design.reset()
        sa_wrapper.design.eval_reset_clocked()
        result = sa_wrapper.compute_tile(a, b, handle.create_callback())

    # IndexError means design produced too many values
    except (HangError, IndexError) as e:
        status = str(e)

    status = "okay" if np.allclose(result, golden) else "SDC"
    results.append({"cell_name": cell_name, "status": status})

In [ ]:
pl.DataFrame(results).group_by("status").agg(pl.len())